<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 11 · Hồi quy logistic</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Mô hình phân loại đầu tiên: sigmoid, cross-entropy, gradient (p - y)x tự cài bằng tay, rồi softmax cho nhiều lớp.</p>
</div>

### Sau notebook này bạn sẽ

- Hiểu vì sao phân loại cần sigmoid và cross-entropy.
- Tự dẫn và kiểm gradient, tự cài hồi quy logistic.
- Đọc xác suất, ranh giới quyết định, tỷ lệ cược.
- Dùng softmax và LogisticRegression cho nhiều lớp.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch11-hoi-quy-logistic/01-bai-toan-phan-loai/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Bài toán phân loại và hàm sigmoid](#trang-1)
2. [Trang 2 · Hàm mất mát cross-entropy](#trang-2)
3. [Trang 3 · Gradient và tự cài](#trang-3)
4. [Trang 4 · scikit-learn và nhiều lớp](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Bài toán phân loại và hàm sigmoid

Dữ liệu đỗ trượt và hàm sigmoid. Code của trang này nằm chung với trang 2.

*Đọc kèm trên web: [Trang 1 · Bài toán phân loại và hàm sigmoid](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch11-hoi-quy-logistic/01-bai-toan-phan-loai/)*

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Hàm mất mát cross-entropy

Cross-entropy và giá trị lúc bắt đầu.

*Đọc kèm trên web: [Trang 2 · Hàm mất mát cross-entropy](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch11-hoi-quy-logistic/02-cross-entropy/)*

In [ ]:
import numpy as np                                    # thư viện tính toán
gio = np.array([1, 2, 2, 3, 3, 4, 4, 5, 6, 7.0])      # Bảng 11.1: giờ học
do = np.array([0, 0, 1, 0, 0, 1, 0, 1, 1, 1.0])       # 1 là đỗ, 0 là trượt
sig = lambda z: 1 / (1 + np.exp(-z))                  # hàm sigmoid

def bce(w, b):                                        # cross-entropy nhị phân (11.2)
    p = sig(w * gio + b)                              # xác suất đỗ đoán ra cho từng bạn
    return float(-np.mean(do * np.log(p) + (1 - do) * np.log(1 - p)))   # trung bình mức phạt

print(round(bce(0, 0), 4), np.log(2))                 # lúc bắt đầu: đoán 50-50 cho mọi bạn

Output mong đợi:

```text
0.6931 0.6931471805599453
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Loss đứng yên ở 0,693 nghĩa là mô hình vẫn đang đoán 50-50, chưa học được gì.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Gradient và tự cài

Kiểm gradient bằng sai phân, rồi tự cài.

*Đọc kèm trên web: [Trang 3 · Gradient và tự cài](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch11-hoi-quy-logistic/03-gradient-va-tu-cai/)*

In [ ]:
def grad(w, b):                                                     # gradient (11.3)
    p = sig(w * gio + b)                                            # xác suất đoán ra
    return float(np.mean((p - do) * gio)), float(np.mean(p - do))   # (đoán trừ thật) nhân đặc trưng

h = 1e-5                                                            # bước sai phân
print(grad(0.3, -1.0))                                              # công thức
print(((bce(0.3 + h, -1) - bce(0.3 - h, -1)) / (2 * h), (bce(0.3, -1 + h) - bce(0.3, -1 - h)) / (2 * h)))  # sai phân

Output mong đợi:

```text
(-0.23101772770510198, 0.024684044519859026)
(-0.23101772779754445, 0.024684044519496236)
```

In [ ]:
w = b = 0.0                                           # khởi tạo
for buoc in range(5001):                              # 5000 bước
    if buoc in (0, 1, 100, 1000, 5000):               # in vài bước
        print(f"Bước {buoc:4d} | w = {w:.4f} | b = {b:.4f} | loss = {bce(w, b):.4f}")  # trạng thái
    gw, gb = grad(w, b)                               # gradient
    w -= 0.1 * gw                                     # learning rate 0,1
    b -= 0.1 * gb                                     # cùng bước cho b
print(round(-b / w, 4))                               # ranh giới quyết định

Output mong đợi:

```text
Bước    0 | w = 0.0000 | b = 0.0000 | loss = 0.6931
Bước    1 | w = 0.0550 | b = 0.0000 | loss = 0.6693
Bước  100 | w = 0.3426 | b = -0.9249 | loss = 0.5633
Bước 1000 | w = 0.9144 | b = -3.2546 | loss = 0.4696
Bước 5000 | w = 1.0204 | b = -3.6696 | loss = 0.4680
3.5962
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Đổi learning rate thành 1.0. Loss có xuống nhanh hơn không?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Đổi nhãn bạn học 4 giờ đang trượt thành đỗ, chạy lại. Ranh giới dịch về đâu?</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · scikit-learn và nhiều lớp

scikit-learn, softmax và bộ hoa iris.

*Đọc kèm trên web: [Trang 4 · scikit-learn và nhiều lớp](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch11-hoi-quy-logistic/04-scikit-learn-va-softmax/)*

In [ ]:
from sklearn.linear_model import LogisticRegression     # mô hình có sẵn
X1 = gio.reshape(-1, 1)                                 # bảng một cột
m = LogisticRegression(C=np.inf).fit(X1, do)            # tắt regularization để so với bản tự cài
print(m.coef_.round(4), m.intercept_.round(4))          # w và b
print(m.predict_proba(np.array([[2.0], [3.5], [5.0]]))[:, 1].round(4))   # xác suất đỗ
print(m.predict(X1), m.score(X1, do))                   # nhãn đoán ra, độ chính xác

Output mong đợi:

```text
[[1.0216]] [-3.6745]
[0.1637 0.4753 0.8075]
[0. 0. 0. 0. 0. 1. 1. 1. 1. 1.] 0.8
```

In [ ]:
z = np.array([2.0, 1.0, 0.1])                 # điểm số ba lớp
e = np.exp(z)                                 # lấy mũ
print(e.round(4), round(e.sum(), 4), (e / e.sum()).round(4))   # softmax (11.4)
big = np.array([1000.0, 999.0])               # điểm số rất lớn
with np.errstate(all="ignore"):               # tắt cảnh báo tràn số để in gọn
    print(np.exp(big) / np.exp(big).sum())    # tràn: ra nan
b2 = big - big.max()                          # trừ số lớn nhất trước
print((np.exp(b2) / np.exp(b2).sum()).round(4))   # không tràn

Output mong đợi:

```text
[7.3891 2.7183 1.1052] 11.2125 [0.659  0.2424 0.0986]
[nan nan]
[0.7311 0.2689]
```

In [ ]:
from sklearn.datasets import load_iris                       # 150 bông hoa, 3 loài
from sklearn.model_selection import train_test_split         # chia dữ liệu
X, y = load_iris(return_X_y=True)                            # 4 đặc trưng
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)   # giữ tỷ lệ 3 loài
m3 = LogisticRegression(max_iter=1000).fit(Xtr, ytr)         # softmax tự động khi có 3 lớp
print(m3.coef_.shape, round(m3.score(Xte, yte), 4))          # 3 x 4 trọng số; độ chính xác test
print(m3.predict_proba(Xte[:3]).round(4), m3.predict(Xte[:3]), yte[:3])   # 3 bông đầu

Output mong đợi:

```text
(3, 4) 1.0
[[0.000e+00 3.480e-02 9.651e-01]
 [5.000e-04 2.333e-01 7.662e-01]
 [9.660e-01 3.400e-02 0.000e+00]] [2 2 0] [2 2 0]
```

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Tính tay trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch11-hoi-quy-logistic/05-bai-tap/)*

### Bài 11.1 · Bước đầu tiên, tính tay

Từ $w = b = 0$, tính gradient và $w$, $b$ sau một bước với learning rate 0,1.
<details><summary>Xem đáp án</summary>

Gradient $(-0{,}55;\ 0)$: năm đỗ năm trượt nên theo $b$ triệt tiêu. Sau bước: $w = 0{,}055$, $b = 0$. Code ở ô ngay dưới.
</details>

In [ ]:
print(grad(0.0, 0.0))          # gradient tại điểm xuất phát

Output mong đợi:

```text
(-0.55, 0.0)
```

### Bài 11.2 · Hạ ngưỡng

Đoán đỗ khi $p > 0{,}3$ thay vì 0,5. Bạn nào đổi kết quả, độ chính xác thành bao nhiêu?
<details><summary>Xem đáp án</summary>

Hai bạn học 3 giờ (xác suất 0,352) chuyển sang đoán đỗ, nhưng cả hai thật ra trượt: độ chính xác giảm từ 0,8 xuống 0,6. Code ở ô ngay dưới.
</details>

In [ ]:
p = sig(1.0216 * gio - 3.6745)                              # xác suất của mô hình trang 4
print((p > 0.3).astype(int), float(((p > 0.3) == do).mean()))   # nhãn với ngưỡng 0,3 và độ chính xác

Output mong đợi:

```text
[0 0 0 1 1 1 1 1 1 1] 0.6
```

### Bài 11.3 · Softmax tính tay

Tính softmax của $(1, 2, 3)$, của $(101, 102, 103)$ và của $(2, 4, 6)$.
<details><summary>Xem đáp án</summary>

$(0{,}090;\ 0{,}245;\ 0{,}665)$. Cộng cùng một số: không đổi. Nhân 2: nhọn hơn, $(0{,}016;\ 0{,}117;\ 0{,}867)$. Code ở ô ngay dưới.
</details>

In [ ]:
sm = lambda v: np.exp(v - v.max()) / np.exp(v - v.max()).sum()   # softmax ổn định số
z = np.array([1.0, 2.0, 3.0])                                    # điểm số
print(sm(z).round(3), sm(z + 100).round(3), sm(2 * z).round(3))  # gốc, cộng 100, nhân 2

Output mong đợi:

```text
[0.09  0.245 0.665] [0.09  0.245 0.665] [0.016 0.117 0.867]
```